In [50]:
import os

In [51]:
%pwd

'C:\\Users\\quamr\\OneDrive\\Desktop\\project\\car-plate-detection'

In [52]:

os.chdir(r"C:\Users\quamr\OneDrive\Desktop\project\car-plate-detection")

In [53]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
   root_dir:Path
   source_url:str
   local_data_file:str
   unzip_dir:str

In [54]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

In [55]:
class ConfigurationManager:
  def __init__(
      self,
      config_filepath:Path=CONFIG_FILE_PATH,
      params_filepath:Path=PARAMS_FILE_PATH
  ):
    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)

    create_directories([self.config.artifacts_root])

  def get_data_ingestion_config(self) -> DataIngestionConfig:
    config = self.config.data_ingestion

    create_directories([config.root_dir])

    data_ingestion_config = DataIngestionConfig(
        root_dir = config.root_dir,
        source_url = config.source_url,
        local_data_file = config.local_data_file,
        unzip_dir = config.unzip_dir
    )
    return data_ingestion_config

In [56]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [57]:
class DataIngestion:
  def __init__(self, config: DataIngestionConfig):
    self.config = config

  def download_file(self,) -> str:
    """Fetch data from google drive
    """
    try:
      dataset_url = self.config.source_url
      zip_download_dir = self.config.local_data_file
      os.makedirs("artifacts/data_ingestion", exist_ok=True)
      logger.info(f"downloading data from {dataset_url} to {zip_download_dir}")

      file_id = dataset_url.split("/")[-2]
      prefix = "https://drive.google.com/uc?export=download&id="
      gdown.download(prefix+file_id, zip_download_dir)

      logger.info(f"Downloaded data from {dataset_url} to {zip_download_dir}")

    except Exception as e:
      raise e
  
  def extract_zip_file(self,)-> None:
    """Extract zip file to directory
    """
    unzip_path = Path(self.config.unzip_dir)
    os.makedirs(unzip_path, exist_ok=True)
    with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
     logger.info(f"Extracting zip file: {self.config.local_data_file} to dir: {unzip_path}")
     zip_ref.extractall(unzip_path)
     logger.info(
          f"Extracted zip file: {self.config.local_data_file} to dir: {unzip_path} and size of data is {get_size(unzip_path)}"
      )


In [58]:
try:
  config = ConfigurationManager()
  data_ingestion_config = config.get_data_ingestion_config()

  data_ingestion = DataIngestion(config=data_ingestion_config)
  data_ingestion.download_file()
  data_ingestion.extract_zip_file()
except Exception as e:
  raise e

[2026-03-09 19:22:01,755:INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-03-09 19:22:01,764:INFO:common:yaml file: params.yaml loaded successfully]
[2026-03-09 19:22:01,766:INFO:common:created directory at: artifacts]
[2026-03-09 19:22:01,771:INFO:common:created directory at: artifacts/data_ingestion]


[2026-03-09 19:22:01,771:INFO:2251519087:downloading data from https://drive.google.com/file/d/1eQBQV-1ovhOrEyXDrUGurDYpdnOc0gbG/view?usp=sharing to artifacts/data_ingestion/car-plate-data.zip]


Downloading...
From (original): https://drive.google.com/uc?export=download&id=1eQBQV-1ovhOrEyXDrUGurDYpdnOc0gbG
From (redirected): https://drive.google.com/uc?export=download&id=1eQBQV-1ovhOrEyXDrUGurDYpdnOc0gbG&confirm=t&uuid=58231829-3064-486a-842c-587cf56237af
To: C:\Users\quamr\OneDrive\Desktop\project\car-plate-detection\artifacts\data_ingestion\car-plate-data.zip
100%|██████████| 213M/213M [01:41<00:00, 2.09MB/s] 

[2026-03-09 19:23:47,208:INFO:2251519087:Downloaded data from https://drive.google.com/file/d/1eQBQV-1ovhOrEyXDrUGurDYpdnOc0gbG/view?usp=sharing to artifacts/data_ingestion/car-plate-data.zip]


[2026-03-09 19:23:47,283:INFO:2251519087:Extracting zip file: artifacts/data_ingestion/car-plate-data.zip to dir: artifacts\data_ingestion]
[2026-03-09 19:23:50,870:INFO:2251519087:Extracted zip file: artifacts/data_ingestion/car-plate-data.zip to dir: artifacts\data_ingestion and size of data is ~ 0 KB]
